<img src="../../img/backdrop-wh.png" alt="Drawing" style="width: 300px;"/>

# LLMs as Coding Collaborators

* * *


<div class="alert alert-success">

### Learning Objectives

* Use an LLM to adapt existing Python code for a computational text-analysis task.
* Write a prompt that includes research context, dataframe information, task constraints, and desired output.
* Recognize when an LLM has introduced hidden assumptions, invented variables, changed the method, or overcomplicated the code.
* Explain, in plain language, what AI-generated code does.
* Verify generated code through manual inspection, debugging, and comparison with the original dataset.
* Reflect on AI-assisted coding as a form of computational interpretation.

</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>

### Sections
1. [Orientation](#orient)
2. [Privacy and Tool Choice](#privacy)
3. [Load and Inspect the Dataset](#load)
4. [Choose a Starting Method](#method)
5. [Run the Original Method](#run)
6. [Prompt Design](#prompt)
7. [LLM Interaction Log](#log)
8. [Paste, Run, and Debug](#debug)
9. [Explain the Code in Plain Language](#explain)
10. [Verify the Output](#verify)
11. [Interpretive Analysis](#interpret)
12. [Peer Comparison Preparation](#peer)
13. [Final Notebook Reflection](#final)


<a id='orient'></a>

In this notebook, you will use a large language model to help modify a text-analysis workflow. The idea is to study what happens when a model helps produce the code through which interpretation happens. The goal is to understand how prompts, generated code, errors, defaults, and outputs shape the interpretive choices you make as a researcher.

**Core principle, repeated throughout this notebook:**

> AI-generated code is allowed. Unexplained code is not.

You remain fully responsible for everything in your submission — what the code does, why it does it, and what interpretive claims you draw from its output.

### What to save as you work

This notebook prepares you directly for the **Week 5 AI Coding Audit** Discussion. Save the following as you go:

- Your prompts (copy them into the markdown cells provided).
- The LLM's main response or a relevant excerpt.
- Any errors you encountered and how you fixed them.
- Your explanation of what the code does.
- Your output and your verification notes.

<a id='privacy'></a>

# 1. Privacy and Tool Choice

Before you interact with any external AI system, there are two practical questions to answer: what tool will you use, and what data is safe to share?

### Privacy

⚠️ **Warning:** Do not paste private, sensitive, identifying, or unpublished data into an AI system. This includes personal data about real individuals, data you collected under IRB constraints, or data you do not have rights to redistribute.

Course-provided Reddit data is acceptable to use. You almost never need to paste the full dataset. Instead:

- Describe your dataframe (column names, number of rows, what one row represents).
- Paste one or two representative rows only if the model genuinely needs them to help with code.
- Do not paste your entire CSV.

### Tool recommendations

| Tool | Notes |
|------|-------|
| **UC-licensed Gemini** | Recommended default. Campus-supported, no paid account required. Access at [gemini.google.com](https://gemini.google.com). Use your Berkeley login. |
| **ChatGPT** | Acceptable. Free tier is sufficient for this assignment. |
| **Claude** | Acceptable. Free tier is sufficient. |
| **GitHub Copilot** | Allowed but not recommended. The prompt-response trail is harder to document, which makes the Week 6 audit difficult. |

No paid subscription is required or expected for this assignment.


### Student Action: Tool Choice

*Replace this text with your answer.*

**Which AI tool are you using for this notebook?**

**Why did you choose it?**

**Are there any privacy or access concerns you need to keep in mind with this tool?**


<a id='load'></a>

# 2. Load and Inspect the Dataset

Ground your LLM task in an actual dataset before you ask the model for anything. This section ensures you know what you are working with before you involve an AI.

### Note on package installation

All packages used in this notebook were installed as part of the course Conda environment. If you are on DataHub and see an ImportError, uncomment and run the pip install line in the cell below, then restart your kernel.


In [ ]:
# All packages are part of the course conda environment
# Uncomment only if you see an ImportError on DataHub:
# %pip install pandas scikit-learn gensim spacy

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Load your selected Reddit dataset below. If you have a preprocessed file from the Week 1 lesson (`aita_pp.csv`), you can use that. Otherwise, load the original submissions file.

💡 **Tip:** If you are working with your own dataset from a different subreddit, update the file path accordingly.


In [ ]:
# Load your dataset — update path if using a different file
# Option 1: preprocessed file (from Week 1 lesson)
try:
    df = pd.read_csv('../../data/aita_pp.csv')
    print("Loaded preprocessed file.")
except FileNotFoundError:
    # Option 2: original submissions
    df = pd.read_csv('../../data/aita_top_submissions.csv')
    print("Loaded original submissions file.")

df = df.dropna(subset=['selftext'])
print(f"Shape: {df.shape}")
df.head(2)

In [ ]:
# Inspect column names
print("Columns:", list(df.columns))
print(f"\nNumber of rows: {len(df)}")
print(f"Text column preview:")
print(df['selftext'].iloc[0][:300])

🔔 **Question:** What does one row in your dataset represent? What text column will you be analyzing?

### Student Action: Dataset Description

*Replace this text with your answer (3–5 sentences).*

**What is your dataset?**

**What is one interpretive question this dataset might help you explore?**

<a id='method'></a>

# 3. Choose a Starting Method

Rather than asking the LLM to build an analysis from scratch, you will ask it to *modify a known method*. Constraining the task this way makes the AI's changes more visible and easier to explain.

Choose one of the following tracks based on your research question:

---

### Track A: Distinctive Language (TF-IDF)
**Best for:** Questions about recurring terms, rhetorical patterns, community vocabulary, or contrastive language across groups (e.g., "What words distinguish YTA posts from NTA posts?").

Start from: TF-IDF or word-frequency analysis (Week 2 lesson).

---

### Track B: Topic Modeling
**Best for:** Questions about recurring themes, discursive formations, or patterns across many posts (e.g., "What kinds of situations dominate this community's moral reasoning?").

Start from: LDA topic modeling (Week 3 lesson).

---

### Track C: Similarity and Examples
**Best for:** Questions about semantic concepts, finding example posts, or studying how posts cluster around a theme (e.g., "Which posts most resemble each other in how they frame boundary violations?").

Start from: TF-IDF vectorization with cosine similarity, or the embedding-based analysis from Week 4.

---

### Student Action: Track Selection

*Replace this text with your answer.*

**Which track are you choosing (A, B, or C)?**

**Why is this method appropriate for your interpretive question?**

**What might this method miss or distort?**


<a id='run'></a>

# 4. Run the Original Method

Run the starter code for your chosen track below. **Do not modify it yet.** This is your baseline, i.e. the output you will compare with the AI-modified version.

After running, scroll past all three tracks to the reflection prompt.

⚠️ **Warning:** Only run the track you chose. You do not need to run all three.


## Track A: TF-IDF Starter

Run this if you chose **Track A: Distinctive Language**.


In [ ]:
# ── TRACK A: TF-IDF STARTER ──────────────────────────────────────────────────
# Run this as your baseline before asking the LLM to modify it.

from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# Vectorize posts
vectorizer_a = TfidfVectorizer(max_features=1000, stop_words='english', max_df=0.85, min_df=5)
tfidf_matrix_a = vectorizer_a.fit_transform(df['selftext'].fillna(''))
feature_names_a = vectorizer_a.get_feature_names_out()

# Compute mean TF-IDF score per term across all documents
mean_scores_a = np.asarray(tfidf_matrix_a.mean(axis=0)).flatten()
top_idx_a = mean_scores_a.argsort()[::-1][:20]
top_terms_a = [(feature_names_a[i], round(float(mean_scores_a[i]), 4)) for i in top_idx_a]

print("Top 20 terms by average TF-IDF score across all posts:\n")
for term, score in top_terms_a:
    print(f"  {term:25s} {score}")

In [ ]:
# Visualize the top terms
terms, scores = zip(*top_terms_a)
plt.figure(figsize=(10, 5))
plt.barh(list(terms)[::-1], list(scores)[::-1])
plt.xlabel("Mean TF-IDF score")
plt.title("Top 20 Terms by Average TF-IDF (all posts)")
plt.tight_layout()
plt.show()

print("\nBaseline output saved. Note what you see before involving the LLM.")

## Track B: Topic Modeling Starter

Run this if you chose **Track B: Topic Modeling**.


In [ ]:
# ── TRACK B: TOPIC MODELING STARTER ─────────────────────────────────────────
# Run this as your baseline before asking the LLM to modify it.

from gensim import corpora
from gensim.models.ldamodel import LdaModel
import warnings
warnings.filterwarnings('ignore')

# Decide which text column to tokenize
text_col_b = 'pp_text' if 'pp_text' in df.columns else 'selftext'
print(f"Using column: '{text_col_b}'")

# Tokenize
tokens_b = [str(doc).split() for doc in df[text_col_b].dropna() if len(str(doc).split()) > 5]
print(f"Documents after filtering: {len(tokens_b)}")

# Build dictionary and corpus
dictionary_b = corpora.Dictionary(tokens_b)
dictionary_b.filter_extremes(no_below=5, no_above=0.5, keep_n=5000)
corpus_b = [dictionary_b.doc2bow(doc) for doc in tokens_b]
print(f"Dictionary size: {len(dictionary_b)} terms")

In [ ]:
# Train LDA model — may take 1-2 minutes
lda_model_b = LdaModel(
    corpus=corpus_b,
    id2word=dictionary_b,
    num_topics=5,
    passes=5,
    random_state=42,
    alpha='auto'
)

print("Topics discovered:\n")
for idx, topic in lda_model_b.show_topics(num_topics=5, num_words=10, formatted=False):
    words = [word for word, _ in topic]
    print(f"  Topic {idx}: {', '.join(words)}")

print("\nBaseline output saved. Note what you see before involving the LLM.")

## Track C: Similarity Starter

Run this if you chose **Track C: Similarity and Examples**.


In [ ]:
# ── TRACK C: SIMILARITY STARTER ──────────────────────────────────────────────
# Run this as your baseline before asking the LLM to modify it.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Limit to 1000 posts for speed — adjust if needed
df_c = df.dropna(subset=['selftext']).head(1000).reset_index(drop=True)

# Vectorize
vectorizer_c = TfidfVectorizer(max_features=500, stop_words='english', min_df=3)
matrix_c = vectorizer_c.fit_transform(df_c['selftext'].fillna(''))

print(f"Matrix shape: {matrix_c.shape}")
print("(rows = posts, columns = terms)\n")

# Example: find posts similar to a short query
query_c = "My roommate keeps eating my food without asking"
query_vec_c = vectorizer_c.transform([query_c])
scores_c = cosine_similarity(query_vec_c, matrix_c).flatten()
top_idx_c = scores_c.argsort()[::-1][:5]

print(f"Posts most similar to: '{query_c}'\n")
for rank, i in enumerate(top_idx_c, 1):
    print(f"  [{rank}] Score {scores_c[i]:.3f} — {df_c['selftext'].iloc[i][:120]}...\n")

In [ ]:
# Examine the similarity matrix (post-to-post) for a small sample
sample_matrix = matrix_c[:50]
sim_matrix = cosine_similarity(sample_matrix)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 6))
plt.imshow(sim_matrix, cmap='viridis', vmin=0, vmax=0.5)
plt.colorbar(label='Cosine Similarity')
plt.title("Post-to-Post Similarity (first 50 posts)")
plt.xlabel("Post index")
plt.ylabel("Post index")
plt.tight_layout()
plt.show()

print("\nBaseline output saved. Note what you see before involving the LLM.")

### Student Action: Document the Baseline

Before continuing, take a moment to record what the original method produced.

💡 **Tip:** Screenshot or copy-paste the key output into a note. You will compare it with the LLM-modified output in Section 9.

💭 **Reflection:** What does the original code do? What does the output show? What is one useful pattern or limitation you notice *before* involving the LLM?

*Replace this text with your answer (3–5 sentences).*


<a id='prompt'></a>

# 5. Prompt Design

The quality of the code you get from an LLM depends heavily on how you ask. A vague request produces vague code. A specific, bounded request produces something you can actually inspect and explain.

### What a strong prompt includes

A strong prompt for this assignment should tell the model:

1. **Your research question** — what are you trying to find out?
2. **The dataframe name and text column** — e.g., `df`, column `selftext`.
3. **What one row represents** — e.g., "one Reddit post from r/AmItheAsshole".
4. **The current code or method** — paste the relevant section or describe it.
5. **The specific modification you want** — be precise about what should change.
6. **Constraints on libraries** — e.g., "use only pandas, scikit-learn, and matplotlib".
7. **A request for comments or explanations** — ask it to explain each step.
8. **A boundary on scope** — "do not rewrite the entire project".

### Strong vs. weak prompts

| Weak prompt | Strong prompt |
|---|---|
| "Make this better." | "Modify the TF-IDF code below to compare top terms between two groups: posts flaired NTA and posts flaired YTA. Use pandas and scikit-learn only. Add a comment explaining each step." |
| "Analyze this dataset." | "I have a dataframe called `df` with 16,000 Reddit posts in column `selftext`. I want to find which 5-word phrases appear most often. Modify the existing CountVectorizer code to use n-gram range (2,5). Explain what each parameter does." |
| "Find interesting patterns." | "The LDA model below produces 5 topics. Modify it to produce 8 topics and add code to print the 3 most representative posts for each topic. Do not change the preprocessing steps." |
| "Write code for my project." | "I am studying how r/AITA posts frame conflicts between parents and adult children. Here is my current similarity search code. Modify the query string to search for posts about financial dependence and return the top 10 results with their post titles." |

### Core principle

> AI-generated code is allowed. Unexplained code is not.

If you paste code from an LLM and cannot explain what every line does, you are not done yet.

### Student Action: Draft Your Prompt

*Replace this text with your first prompt before you send it to the LLM.*

**Your prompt:**

```
[Paste your prompt here]
```

**What do you expect the model to produce?**


<a id='log'></a>

# 6. LLM Interaction Log

After sending your prompt, document the model's response here before running anything.

This log becomes the raw material for your Week 6 AI Coding Audit.

### Student Action: Paste or Summarize the Response

*Replace this text with the model's main response or a relevant excerpt.*

**Main response (paste excerpt or summarize):**

**What did the model propose changing?**

- [ ] Added new libraries
- [ ] Changed the method entirely
- [ ] Introduced new variables or columns
- [ ] Introduced new parameters or settings
- [ ] Added visualization
- [ ] Added comments or explanations
- [ ] Other: _______________

**Was the response usable?**

- [ ] Fully usable as written
- [ ] Partially usable — needed edits
- [ ] Too complex or unfamiliar
- [ ] Wrong — misunderstood the task
- [ ] Unclear — needed follow-up prompt

**Did the model stay within your prompt constraints?** If not, note what it introduced without being asked.

💭 **Reflection:** What did the model suggest changing? Did it introduce any new variables, libraries, assumptions, or analytic steps you did not ask for?


💡 **Tip:** If the first response was not useful, write a follow-up prompt. Paste that here too, and note what changed in the second response. Iteration is normal and worth documenting — it shows the collaboration process, which is exactly what the Week 6 audit asks you to analyze.

**Follow-up prompt (if applicable):**

```
[Paste follow-up prompt here, or delete this block if you did not need one]
```

**What changed in the follow-up response?**


<a id='debug'></a>

# 7. Paste, Run, and Debug

Paste the AI-generated code into the cell below and run it.

⚠️ **Warning:** Do not run code you do not understand. Read through it first. If the model introduced libraries you have not seen, look them up before running.

⚠️ **Warning:** Failure is expected and normal. An error is not a failed assignment. An undocumented error is the problem. If the code fails, record the error message and what you changed to fix it.


In [ ]:
# ── PASTE AI-GENERATED CODE HERE ─────────────────────────────────────────────
# Read through the code before running it.
# Add your own comments to explain anything you did not understand at first.

# [paste code here]


### Student Action: Debug Log

*Replace this text with your debugging notes.*

**Did the code run on the first try?**

**If not, what error message appeared?**

```
[paste error here]
```

**What did you change to fix it?**

**Did the model misunderstand anything about your dataset or method?** (e.g., wrong column name, assumed a column that does not exist, used a library not in your environment)

💭 **Reflection:** What did you learn from the error or from editing the generated code?


<a id='explain'></a>

# 8. Explain the Code in Plain Language

Now that the code runs, explain what it does. Do not paraphrase comments written by the model. Write the explanation in your own words, as if you were explaining it to a classmate who has taken this course but has not seen your notebook.

A complete explanation covers:

- **Input:** What does the code take as input? What column? How many rows?
- **Transformation:** What does the code do to the text? What algorithm does it use? What choices does it make (e.g., number of topics, vocabulary size, similarity threshold)?
- **Output:** What does the code produce? What do the rows or columns in the output mean?
- **Interpretive consequences:** What can you conclude from this output? What is it measuring?

🔔 **Question:** Can you explain every line of the code you pasted? If there is a line you cannot explain, look it up or ask the LLM to explain it — and then add that explanation to your notes.

### Student Action: Plain-Language Explanation

*Replace this text with your explanation.*

**Input:**

**Transformation:**

**Output:**

**Interpretive consequences:**


<a id='verify'></a>

# 9. Verify the Output

Code that runs without errors is not necessarily correct. This section asks you to check whether the output actually reflects what the model said it would produce, and whether it makes sense against the original data.

**Perform at least two of the following verification steps:**

- Inspect several rows manually.
- Check that expected columns exist.
- Compare the output with the baseline from Section 4.
- Check whether extracted examples actually contain the terms or patterns the output claims.
- Try one alternate parameter value (e.g., different number of topics, different query, different vocabulary size).
- Check whether labels or topic keywords make sense against actual post text.
- Confirm that the model did not hallucinate column names, category labels, or results.


In [ ]:
# ── VERIFICATION STEP 1 ───────────────────────────────────────────────────────
# Manually inspect a sample of the output
# Replace this with verification code appropriate to your track

# Example for Track A: check a high-TF-IDF term actually appears in posts
# term_to_check = 'example_term'
# matching = df[df['selftext'].str.contains(term_to_check, case=False, na=False)]
# print(f"Posts containing '{term_to_check}': {len(matching)}")
# print(matching['selftext'].iloc[0][:400])

# Example for Track B: check what a topic's actual posts look like
# topic_num = 0
# topic_posts = [(i, lda_model_b.get_document_topics(corpus_b[i])) for i in range(len(corpus_b))]
# top_posts = sorted(topic_posts, key=lambda x: dict(x[1]).get(topic_num, 0), reverse=True)[:3]
# for idx, _ in top_posts:
#     print(tokens_b[idx][:20])

# Example for Track C: verify top similarity result actually matches the query
# print("Query:", query_c)
# print("\nTop result:")
# print(df_c['selftext'].iloc[top_idx_c[0]])

print("Replace this cell with your actual verification code.")

In [ ]:
# ── VERIFICATION STEP 2 ───────────────────────────────────────────────────────
# Compare with baseline from Section 4, or test an alternate parameter

print("Replace this cell with your second verification step.")

### Student Action: Verification Notes

*Replace this text with your answers.*

**What did you do to check the result?**

**What became more trustworthy after checking?**

**What remains uncertain or unclear about the output?**

💭 **Reflection:** Did the LLM's code produce what it claimed to produce? Did checking the output change how you interpret the results?


<a id='interpret'></a>

# 10. Interpretive Analysis

Now connect the technical workflow back to your research question.

This section is the interpretive core of the notebook. The goal is not just to describe what the code does, but to think critically about what the AI-assisted analysis made visible, what it may have hidden, and how it shaped your interpretation.

Your analysis should address:

- What patterns does the output reveal in the dataset?
- Did the LLM sharpen your question, change your method, introduce categories, or mainly help with implementation?
- What assumptions about language, meaning, similarity, topics, or evidence entered through the generated code?
- Is there something the output *cannot* tell you about the posts?

💭 **Reflection:** What did the AI-assisted workflow help you see in the dataset? What assumptions about language, meaning, similarity, topics, or evidence entered through the generated code?

### Student Action: Write 250–350 words

*Replace this text with your interpretive analysis.*


<a id='peer'></a>

# 11. Peer Comparison Preparation

Before completing this notebook, exchange the following artifacts with a peer:

| Artifact | You share | Peer shares |
|---|---|---|
| Research question | ✓ | ✓ |
| Main prompt (from Section 5) | ✓ | ✓ |
| One code excerpt (from Section 7 or 8) | ✓ | ✓ |
| One output (screenshot or text) | ✓ | ✓ |
| Plain-language explanation (from Section 8) | ✓ | ✓ |

You will use this comparison in the Week 6 AI Coding Audit.

### Student Action: Record Peer Artifacts

**Peer's name:**

**Peer's research question:**

**Peer's main prompt:**

```
[paste or summarize]
```

**Peer's code approach (brief description):**

**What was similar between your workflow and your peer's?**

**What differed in your prompts, generated code, outputs, or interpretations?**

💭 **Reflection:** Did the LLM push your analyses in similar or different directions? What does the comparison reveal about how prompting shapes computational interpretation?


<div class="alert alert-success">

## ❗ Key Points

* AI-generated code is allowed. Unexplained code is not.
* A strong LLM prompt includes your research question, dataframe structure, current method, specific modification, library constraints, and a request for comments.
* LLMs regularly introduce hidden assumptions: new variables, changed methods, overcomplicated logic, or hallucinated column names.
* Debugging and verification are interpretive labor, not optional steps.
* The comparison between original and AI-modified output is where interpretation happens.
* Save your prompts, responses, errors, and explanations — you will need them for the Week 6 AI Coding Audit.

</div>
